# Setup

In [40]:
import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from tqdm.auto import tqdm

from torchvision.models import alexnet

import kagglehub

In [41]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [43]:
import os

SAVE_DIR = "./saved_models_small_kernels"
os.makedirs(SAVE_DIR, exist_ok=True)

## Dataset and trasnforms

In [44]:
dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")

In [45]:
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(64, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

transform_val = transforms.Compose([
    transforms.Resize(64),
    transforms.CenterCrop(64),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

In [46]:
full_dataset = datasets.ImageFolder(train_path)

train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_idx, val_idx = random_split(
    range(len(full_dataset)),
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataset = datasets.ImageFolder(train_path, transform=transform_train)
val_dataset = datasets.ImageFolder(train_path, transform=transform_val)

train_dataset = torch.utils.data.Subset(train_dataset, train_idx.indices)
val_dataset = torch.utils.data.Subset(val_dataset, val_idx.indices)

In [47]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

# Model Definition

## Baseline CNN

In [48]:
class SmallKernelCNN(nn.Module):
    def __init__(self, num_classes=200):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)

## Small Kernel ResNet (3×3 residual network)

In [49]:
class SKResidualBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(c, c, 3, padding=1, bias=False),
            nn.BatchNorm2d(c),
            nn.ReLU(),
            nn.Conv2d(c, c, 3, padding=1, bias=False),
            nn.BatchNorm2d(c),
        )

    def forward(self, x):
        return torch.relu(x + self.block(x))


class SmallKernelResNet(nn.Module):
    def __init__(self, num_classes=200):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.stage = nn.Sequential(
            SKResidualBlock(64),
            SKResidualBlock(64),
            SKResidualBlock(64),

            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            SKResidualBlock(128),
            SKResidualBlock(128),

            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            SKResidualBlock(256),
            SKResidualBlock(256),
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.head(self.stage(self.stem(x)))

## 2×2 Kernel Network (Winograd edge case model)

In [50]:
class Kernel2x2CNN(nn.Module):
    def __init__(self, num_classes=200):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 2, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, 2, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 256, 2, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)

## Mixed Small-Kernel Depth Network (3×3 + 2×2 comparison)

In [51]:
class MixedSmallKernelNet(nn.Module):
    def __init__(self, num_classes=200):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, 2, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 256, 2, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# Training

In [52]:
def train_model(model, train_loader, val_loader, epochs, optimizer, scheduler, criterion, save_name=None):
    scaler = torch.amp.GradScaler("cuda")
    model.to(device)

    history = {
        "train_acc": [],
        "val_acc": [],
        "train_loss": []
    }

    best_acc = 0

    for epoch in range(epochs):
        model.train()

        correct, total = 0, 0
        running_loss = 0

        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

        for x, y in loop:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()

            with torch.amp.autocast("cuda"):
                out = model(x)
                loss = criterion(out, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            preds = out.argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)
            running_loss += loss.item()

            loop.set_postfix(acc=100*correct/total, loss=loss.item())

        train_acc = 100 * correct / total
        train_loss = running_loss / len(train_loader)

        model.eval()
        vc, vt = 0, 0

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                preds = out.argmax(1)

                vc += (preds == y).sum().item()
                vt += y.size(0)

        val_acc = 100 * vc / vt

        if scheduler:
            scheduler.step()

        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["train_loss"].append(train_loss)

        print(f"Epoch {epoch+1}: train={train_acc:.2f}% val={val_acc:.2f}%")

        # ✔ BEST MODEL CHECKPOINT
        if val_acc > best_acc:
            best_acc = val_acc
            if save_name is not None:
                torch.save(model.state_dict(), os.path.join(SAVE_DIR, save_name))

    return best_acc, history

In [53]:
models_dict = {
    "SmallKernelCNN": SmallKernelCNN,
    "SmallKernelResNet": SmallKernelResNet,
    "Kernel2x2CNN": Kernel2x2CNN,
    "MixedSmallKernelNet": MixedSmallKernelNet
}

In [54]:
EPOCHS = 30

def build_optimizer(model):
    return optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)

def build_scheduler(optimizer):
    return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

In [55]:
results = {}

for name, model_cls in models_dict.items():

    print("\n" + "="*60)
    print(f"Training {name}")
    print("="*60)

    model = model_cls(num_classes=200).to(device)

    optimizer = build_optimizer(model)
    scheduler = build_scheduler(optimizer)

    acc = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=EPOCHS,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        save_name=f"{name}.pth"
    )

    results[name] = acc


Training SmallKernelCNN


Epoch 1/30:   0%|          | 0/1407 [00:00<?, ?it/s]

Epoch 1/30: 100%|██████████| 1407/1407 [00:23<00:00, 60.85it/s, acc=4.35, loss=4.41]


Epoch 1: train=4.35% val=6.34%


Epoch 2/30: 100%|██████████| 1407/1407 [00:21<00:00, 66.70it/s, acc=8.24, loss=4.13]


...

Epoch 28: train=37.89% val=34.45%


Epoch 29/30: 100%|██████████| 1407/1407 [00:25<00:00, 55.99it/s, acc=38.2, loss=3.24]


Epoch 29: train=38.17% val=34.54%


Epoch 30/30: 100%|██████████| 1407/1407 [00:25<00:00, 56.09it/s, acc=38.2, loss=3.47]


Epoch 30: train=38.18% val=34.65%


# Evaluation and ranking

In [56]:
def evaluate_model(model, loader, device):
    model.to(device)
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            preds = out.argmax(1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return 100.0 * correct / total

In [57]:
final_results = {}

for name in models_dict.keys():

    model = models_dict[name](num_classes=200).to(device)

    state = torch.load(
        os.path.join(SAVE_DIR, f"{name}.pth"),
        map_location=device
    )

    model.load_state_dict(state)

    acc = evaluate_model(model, val_loader, device)
    final_results[name] = acc

    print(f"{name:25s} -> {acc:.2f}%")

/tmp/ipykernel_71950/2081785851.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(


SmallKernelCNN            -> 22.00%
SmallKernelResNet         -> 51.22%
Kernel2x2CNN              -> 16.70%
MixedSmallKernelNet       -> 34.72%


In [58]:
print("\nFINAL RANKING")
print("="*60)

sorted_results = sorted(final_results.items(), key=lambda x: x[1], reverse=True)

for i, (name, acc) in enumerate(sorted_results, 1):
    print(f"{i:02d}. {name:25s} -> {acc:.2f}%")


FINAL RANKING
01. SmallKernelResNet         -> 51.22%
02. MixedSmallKernelNet       -> 34.72%
03. SmallKernelCNN            -> 22.00%
04. Kernel2x2CNN              -> 16.70%


In [59]:
class StandardKernelCNN(nn.Module):
    def __init__(self, num_classes=200):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [60]:
def build_optimizer(model, lr=3e-4, wd=5e-4):
    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

def build_scheduler(optimizer, epochs):
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

In [61]:
SAVE_DIR = "./saved_models_small_kernels"
os.makedirs(SAVE_DIR, exist_ok=True)

In [62]:
standard_epochs = 30

model = StandardKernelCNN(num_classes=200).to(device)

optimizer = build_optimizer(model, lr=3e-4, wd=5e-4)
scheduler = build_scheduler(optimizer, standard_epochs)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

acc, hist = train_model(
    model,
    train_loader,
    val_loader,
    standard_epochs,
    optimizer,
    scheduler,
    criterion
)

print("StandardKernelCNN:", acc)
torch.save(model.state_dict(), os.path.join(SAVE_DIR, "standard_kernel_cnn.pth"))

Epoch 1/30: 100%|██████████| 1407/1407 [00:20<00:00, 69.54it/s, acc=6.19, loss=4.49]


Epoch 1: train=6.19% val=10.64%


Epoch 2/30: 100%|██████████| 1407/1407 [00:20<00:00, 68.55it/s, acc=12.3, loss=4.45]


Epoch 2: train=12.31% val=13.49%


Epoch 3/30: 100%|██████████| 1407/1407 [00:19<00:00, 71.67it/s, acc=15.9, loss=3.91]


...

Epoch 28: train=35.43% val=31.69%


Epoch 29/30: 100%|██████████| 1407/1407 [00:20<00:00, 68.77it/s, acc=35.5, loss=3.74]


Epoch 29: train=35.49% val=31.70%


Epoch 30/30: 100%|██████████| 1407/1407 [00:20<00:00, 68.52it/s, acc=35.5, loss=3.64]


Epoch 30: train=35.54% val=32.26%
StandardKernelCNN: 32.26


In [63]:
resnet_epochs = 80  # increased from baseline (important for fairness)

model = SmallKernelResNet(num_classes=200).to(device)

optimizer = build_optimizer(model, lr=3e-4, wd=5e-4)
scheduler = build_scheduler(optimizer, resnet_epochs)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

acc, hist = train_model(
    model,
    train_loader,
    val_loader,
    resnet_epochs,
    optimizer,
    scheduler,
    criterion
)

print("SmallKernelResNet (extended):", acc)
torch.save(model.state_dict(), os.path.join(SAVE_DIR, "small_kernel_resnet_long.pth"))

Epoch 1/80: 100%|██████████| 1407/1407 [01:46<00:00, 13.17it/s, acc=5.23, loss=4.26]


Epoch 1: train=5.23% val=7.26%


Epoch 2/80: 100%|██████████| 1407/1407 [01:47<00:00, 13.14it/s, acc=12.4, loss=4.18]


Epoch 2: train=12.45% val=14.40%


Epoch 3/80: 100%|██████████| 1407/1407 [01:47<00:00, 13.15it/s, acc=18.1, loss=4.39]


...

Epoch 78: train=82.62% val=55.47%


Epoch 79/80: 100%|██████████| 1407/1407 [01:46<00:00, 13.16it/s, acc=82.6, loss=1.75]


Epoch 79: train=82.58% val=55.25%


Epoch 80/80: 100%|██████████| 1407/1407 [01:46<00:00, 13.17it/s, acc=82.6, loss=1.9] 


Epoch 80: train=82.60% val=55.64%
SmallKernelResNet (extended): 55.8


In [65]:
models = {
    "StandardKernelCNN": StandardKernelCNN(num_classes=200),
    "SmallKernelCNN": SmallKernelCNN(num_classes=200),
    "Kernel2x2CNN": Kernel2x2CNN(num_classes=200),
    "MixedSmallKernelNet": MixedSmallKernelNet(num_classes=200),
    "SmallKernelResNet": SmallKernelResNet(num_classes=200),
}

paths = {
    "StandardKernelCNN": "standard_kernel_cnn.pth",
    "SmallKernelCNN": "SmallKernelCNN.pth",
    "Kernel2x2CNN": "Kernel2x2CNN.pth",
    "MixedSmallKernelNet": "MixedSmallKernelNet.pth",
    "SmallKernelResNet": "SmallKernelResNet.pth",
}

results = {}

print("\nEVALUATION")
print("=" * 50)

for name, model in models.items():
    model.load_state_dict(torch.load(os.path.join(SAVE_DIR, paths[name]), map_location=device))
    acc = evaluate_model(model, val_loader, device)
    results[name] = acc
    print(f"{name:25s}: {acc:.2f}%")

print("\nFINAL RANKING")
print("=" * 50)

for i, (name, acc) in enumerate(sorted(results.items(), key=lambda x: x[1], reverse=True), 1):
    print(f"{i:02d}. {name:25s} -> {acc:.2f}%")


EVALUATION


/tmp/ipykernel_71950/4081704201.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(os.path.join(SAVE_DIR, paths[name]), map_location=devic

StandardKernelCNN        : 32.26%
SmallKernelCNN           : 22.00%
Kernel2x2CNN             : 16.70%
MixedSmallKernelNet      : 34.72%
SmallKernelResNet        : 51.22%

FINAL RANKING
01. SmallKernelResNet         -> 51.22%
02. MixedSmallKernelNet       -> 34.72%
03. StandardKernelCNN         -> 32.26%
04. SmallKernelCNN            -> 22.00%
05. Kernel2x2CNN              -> 16.70%


In [66]:
import torch
import os

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def model_size_mb(model, dtype_size=4):
    # rough estimate: parameters × bytes (FP32 = 4 bytes)
    param_count = sum(p.numel() for p in model.parameters())
    return (param_count * dtype_size) / (1024 ** 2)

In [67]:
print("\nMODEL COMPLEXITY SUMMARY")
print("=" * 60)

complexity = {}

for name, model in models.items():
    model = model.to(device)

    params = count_parameters(model)
    size_mb = model_size_mb(model)

    complexity[name] = {
        "params": params,
        "size_mb": size_mb
    }

    print(f"{name:25s} | Params: {params/1e6:.2f}M | Size: {size_mb:.2f} MB")


MODEL COMPLEXITY SUMMARY
StandardKernelCNN         | Params: 1.09M | Size: 4.14 MB
SmallKernelCNN            | Params: 0.42M | Size: 1.61 MB
Kernel2x2CNN              | Params: 0.22M | Size: 0.83 MB
MixedSmallKernelNet       | Params: 0.96M | Size: 3.65 MB
SmallKernelResNet         | Params: 3.60M | Size: 13.72 MB


In [68]:
results = {}

print("\nEVALUATION")
print("=" * 50)

for name, model in models.items():
    model.load_state_dict(torch.load(os.path.join(SAVE_DIR, paths[name]), map_location=device))
    acc = evaluate_model(model, val_loader, device)
    results[name] = acc
    print(f"{name:25s}: {acc:.2f}%")


EVALUATION


/tmp/ipykernel_71950/3888940457.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(os.path.join(SAVE_DIR, paths[name]), map_location=device

StandardKernelCNN        : 32.26%
SmallKernelCNN           : 22.00%
Kernel2x2CNN             : 16.70%
MixedSmallKernelNet      : 34.72%
SmallKernelResNet        : 51.22%


In [69]:
print("\nFINAL RANKING (ACCURACY + COMPLEXITY)")
print("=" * 70)

sorted_models = sorted(results.items(), key=lambda x: x[1], reverse=True)

for i, (name, acc) in enumerate(sorted_models, 1):
    params = complexity[name]["params"]
    size_mb = complexity[name]["size_mb"]

    print(
        f"{i:02d}. {name:25s} -> {acc:.2f}% | "
        f"{params/1e6:.2f}M params | {size_mb:.2f} MB"
    )


FINAL RANKING (ACCURACY + COMPLEXITY)
01. SmallKernelResNet         -> 51.22% | 3.60M params | 13.72 MB
02. MixedSmallKernelNet       -> 34.72% | 0.96M params | 3.65 MB
03. StandardKernelCNN         -> 32.26% | 1.09M params | 4.14 MB
04. SmallKernelCNN            -> 22.00% | 0.42M params | 1.61 MB
05. Kernel2x2CNN              -> 16.70% | 0.22M params | 0.83 MB


In [70]:
print("\nEFFICIENCY (accuracy / 1M params)")
print("=" * 50)

for name, acc in sorted_models:
    params_m = complexity[name]["params"] / 1e6
    eff = acc / params_m

    print(f"{name:25s}: {eff:.2f} acc/M-param")


EFFICIENCY (accuracy / 1M params)
SmallKernelResNet        : 14.24 acc/M-param
MixedSmallKernelNet      : 36.32 acc/M-param
StandardKernelCNN        : 29.71 acc/M-param
SmallKernelCNN           : 52.05 acc/M-param
Kernel2x2CNN             : 76.99 acc/M-param
